# Analisis Pengaruh Augmentasi Data terhadap Performa YOLOv8 pada Deteksi Penggunaan Helm Pengendara Sepeda Motor Berbasis Web

---

## Identitas Proyek Penelitian
- **Topik Penelitian**: Deteksi Objek Penggunaan Helm Pengendara Sepeda Motor (*Helmet vs. No-Helmet*)
- **Arsitektur Model**: YOLOv8 Nano (`yolov8n.pt`)
- **Fokus Utama**: Analisis komparatif pengaruh teknik augmentasi data (*offline* dan *online*) terhadap performa deteksi
- **Lingkungan Komputasi**: Google Colab (Akselerator GPU NVIDIA Tesla T4 16 GB, CUDA 13.0)
- **Framework & Versi**: Ultralytics YOLOv8 (`8.4.178`), PyTorch (`2.11.0+cu130`), OpenCV (`4.12.0`)
- **Penyimpanan Persisten**: Google Drive (`/content/drive/MyDrive/Penelitian_YOLOv8/`)

---

## Daftar Isi
1. [Bagian 1: Judul dan Identitas Penelitian](#analisis-pengaruh-augmentasi-data-terhadap-performa-yolov8-pada-deteksi-penggunaan-helm-pengendara-sepeda-motor-berbasis-web)
2. [Bagian 2: Tujuan Eksperimen dan Metodologi Skenario (M0–M4)](#bagian-2-tujuan-eksperimen-dan-metodologi-skenario-m0m4)
3. [Bagian 3: Deskripsi Dataset dan Definisi Kelas](#bagian-3-deskripsi-dataset-dan-definisi-kelas)
4. [Bagian 4: Instalasi Dependensi Pustaka](#bagian-4-instalasi-dependensi-pustaka)
5. [Bagian 5: Mount Google Drive dan Konfigurasi Path Persisten](#bagian-5-mount-google-drive-dan-konfigurasi-path-persisten)
6. [Bagian 6: Ekstraksi dan Pemeriksaan Integritas Dataset Sumber](#bagian-6-ekstraksi-dan-pemeriksaan-integritas-dataset-sumber)
7. [Bagian 7: Pembagian Dataset Terstratifikasi (Train, Valid, Test)](#bagian-7-pembagian-dataset-terstratifikasi-train-valid-test)
8. [Bagian 8: Pembuatan Folder Dataset Dasar Skenario M0–M4 dan Berkas data.yaml](#bagian-8-pembuatan-folder-dataset-dasar-skenario-m0m4-dan-berkas-datayaml)
9. [Bagian 9: Pembuatan Augmentasi Offline (M1, M2, dan M3)](#bagian-9-pembuatan-augmentasi-offline-m1-m2-dan-m3)
10. [Bagian 10: Spesifikasi Parameter Augmentasi Online M4 dan Isolasi M0–M3](#bagian-10-spesifikasi-parameter-augmentasi-online-m4-dan-isolasi-m0m3)
11. [Bagian 11: Validasi Komprehensif Seluruh Skenario (M0–M4)](#bagian-11-validasi-komprehensif-seluruh-skenario-m0m4)
12. [Bagian 12: Pemeriksaan Akselerator GPU dan Konfigurasi Hyperparameter](#bagian-12-pemeriksaan-akselerator-gpu-dan-konfigurasi-hyperparameter)
13. [Bagian 13: Konfigurasi Output Persisten Google Drive](#bagian-13-konfigurasi-output-persisten-google-drive)
14. [Bagian 14: Eksekusi Training YOLOv8 (M0 s.d. M4) — Belum Dijalankan](#bagian-14-eksekusi-training-yolov8-m0-sd-m4--belum-dijalankan)
15. [Bagian 15: Evaluasi Model Terbaik pada Split Test Independen — Belum Dijalankan](#bagian-15-evaluasi-model-terbaik-pada-split-test-independen--belum-dijalankan)
16. [Bagian 16: Rekapitulasi Tabel dan Visualisasi Komparasi Metrik Aktual](#bagian-16-rekapitulasi-tabel-dan-visualisasi-komparasi-metrik-aktual)


## Bagian 2: Tujuan Eksperimen dan Metodologi Skenario (M0–M4)

Penelitian ini bertujuan untuk mengukur secara empiris dan kuantitatif kontribusi masing-masing teknik augmentasi data terhadap akurasi dan ketahanan deteksi model YOLOv8n.

### Desain 5 Skenario Eksperimen
1. **Skenario M0 (Baseline Murni)**:
   - *Training Data*: 636 citra asli.
   - *Augmentasi*: **Tanpa augmentasi offline** dan **seluruh augmentasi online bawaan dinonaktifkan** (`mosaic=0.0`, `fliplr=0.0`, dll.).
   - *Fungsi*: Titik tolak dasar (*benchmark*) pembanding performa.
2. **Skenario M1 (Horizontal Flip Offline)**:
   - *Training Data*: 1.272 citra (636 asli + 636 hasil pencerminan horizontal secara offline).
   - *Augmentasi Online*: Dinonaktifkan (`0.0`).
   - *Fungsi*: Mengisolasi efek variasi orientasi arah laju pengendara.
3. **Skenario M2 (Rotasi Terkendali $\pm 5^\circ$ Offline)**:
   - *Training Data*: 1.272 citra (636 asli + 636 hasil rotasi terkontrol $\pm 5^\circ$ secara offline).
   - *Koreksi Geometris*: Transformasi matriks koordinat 4 titik sudut *bounding box* piksel dengan pembatasan batas citra (*clipping*).
   - *Augmentasi Online*: Dinonaktifkan (`0.0`).
   - *Fungsi*: Mengisolasi efek kemiringan sudut kamera dan posisi pengendara.
4. **Skenario M3 (Penyesuaian Brightness & Contrast Offline)**:
   - *Training Data*: 1.272 citra (636 asli + 636 hasil transformasi kecerahan dan kontras $lpha=1.15, eta=15$ secara offline).
   - *Augmentasi Online*: Dinonaktifkan (`0.0`).
   - *Fungsi*: Mengisolasi efek variasi intensitas pencahayaan lingkungan jalan raya.
5. **Skenario M4 (Default Ultralytics Online Augmentation)**:
   - *Training Data*: 636 citra dasar (augmentasi diterapkan secara dinamis saat batch training dimuat).
   - *Augmentasi Online*: Menggunakan parameter bawaan resmi YOLOv8 yang dideklarasikan secara eksplisit (`mosaic=1.0`, `fliplr=0.5`, `scale=0.5`, `translate=0.1`, `hsv_h=0.015`, `hsv_s=0.7`, `hsv_v=0.4`, `erasing=0.4`, `auto_augment='randaugment'`).
   - *Fungsi*: Membandingkan pendekatan augmentasi offline spesifik terhadap strategi augmentasi dinamis bawaan framework.

> **Prinsip Fair Comparison (Bebas Bias):**
> Subset **Validation (137 citra)** dan **Test (135 citra)** dipertahankan **100% identik dan tanpa augmentasi** di seluruh skenario M0 s.d. M4. Bounding box dan citra pada set evaluasi tidak pernah diubah.


## Bagian 3: Deskripsi Dataset dan Definisi Kelas

Dataset penelitian bersumber dari rekaman deteksi helm pengendara motor yang telah dianotasi dalam format standar YOLO (*normalized bounding box* `class_id x_center y_center width height`).

### Spesifikasi Kelas Anotasi
- **ID 0 (`helmet`)**: Pengendara sepeda motor atau penumpang yang mengenakan helm berstandar pengaman secara benar.
- **ID 1 (`no-helmet`)**: Pengendara sepeda motor atau penumpang yang tidak mengenakan helm.

### Distribusi Awal Dataset Sumber
- Sumber: Arsip ZIP `yolov8-helmet-final-use-no-augmentation.v2-no-augmentasi.yolov8.zip`
- Total citra asli: **908 citra** unik (tanpa duplikasi hash SHA-256)
- Rasio pembagian target:
  - **Train**: $pprox 70\%$ (636 citra)
  - **Validation**: $pprox 15\%$ (137 citra)
  - **Test**: $pprox 15\%$ (135 citra)


## Bagian 4: Instalasi Dependensi Pustaka

Bagian ini memastikan pustaka resmi Ultralytics YOLOv8 terpasang pada lingkungan Google Colab.

> **Catatan Pemulihan Runtime:** Jalankan sel ini setiap kali memulai sesi runtime Colab baru.


In [ ]:
# Instalasi pustaka Ultralytics YOLOv8
%pip install ultralytics

# Pemeriksaan versi library utama
import cv2
import yaml
import ultralytics
import torch

print("Status Instalasi Dependensi:")
print("  - Ultralytics:", ultralytics.__version__)
print("  - PyTorch    :", torch.__version__)
print("  - OpenCV     :", cv2.__version__)
print("  - PyYAML     :", yaml.__version__)
print("  - CUDA Tersedia:", torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Status Instalasi Dependensi:
  - Ultralytics: 8.4.198
  - PyTorch    : 2.11.0+cu130
  - OpenCV     : 4.14.0
  - PyYAML     : 6.0.3
  - CUDA Tersedia: True


## Bagian 5: Mount Google Drive dan Konfigurasi Path Persisten

Google Drive digunakan sebagai media penyimpanan persisten (*persistent storage*) untuk menyimpan arsip dataset sumber serta hasil keluaran eksperimen (bobot model terbaik `best.pt`, log training, confusion matrix, dan berkas CSV). Folder `/content/` pada mesin Colab hanya digunakan untuk proses komputasi sementara (*ephemeral workspace*).


In [ ]:
from pathlib import Path
import os

# 1. Mount Google Drive secara otomatis dan aman jika belum terpasang
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
        print("Google Drive berhasil di-mount.")
    else:
        print("Google Drive sudah terhubung sebelumnya.")
except ImportError:
    print("Lingkungan lokal terdeteksi (bukan Google Colab).")

# 2. Definisi path direktori penelitian di Google Drive
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Penelitian_YOLOv8")
ZIP_DATASET_SOURCE = DRIVE_PROJECT_ROOT / "yolov8-helmet-final-use-no-augmentation.v2-no-augmentasi.yolov8.zip"
EXPERIMENT_OUTPUT_DIR = DRIVE_PROJECT_ROOT / "hasil_eksperimen_final_M0_M4"

# 3. Definisi path workspace lokal sementara di Colab (/content/)
LOCAL_EXTRACT_DIR = Path("/content/dataset_helmet")
LOCAL_SPLIT_DIR = Path("/content/dataset_helmet_split")
LOCAL_EXPERIMENT_ROOT = Path("/content/eksperimen_final_M0_M4")

print()
print("Pemeriksaan dataset selesai")
print("  - Root Google Drive     :", DRIVE_PROJECT_ROOT)
print("  - Arsip ZIP Dataset     :", ZIP_DATASET_SOURCE)
print("  - Output Persisten      :", EXPERIMENT_OUTPUT_DIR)
print("  - Workspace Lokal Colab :", LOCAL_EXPERIMENT_ROOT)

if not DRIVE_PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Direktori penelitian di Google Drive tidak ditemukan: {DRIVE_PROJECT_ROOT}\n"
        "Pastikan shortcut atau folder Penelitian_YOLOv8 tersedia pada MyDrive Anda."
    )


Mounted at /content/drive
Google Drive berhasil di-mount.

Pemeriksaan dataset selesai
  - Root Google Drive     : /content/drive/MyDrive/Penelitian_YOLOv8
  - Arsip ZIP Dataset     : /content/drive/MyDrive/Penelitian_YOLOv8/yolov8-helmet-final-use-no-augmentation.v2-no-augmentasi.yolov8.zip
  - Output Persisten      : /content/drive/MyDrive/Penelitian_YOLOv8/hasil_eksperimen_final_M0_M4
  - Workspace Lokal Colab : /content/eksperimen_final_M0_M4


## Bagian 6: Ekstraksi dan Pemeriksaan Integritas Dataset Sumber

Pada tahap ini, arsip dataset diekstraksi ke direktori lokal Colab `/content/dataset_helmet`. Pemeriksaan hash SHA-256 dilakukan untuk menjamin bahwa seluruh 908 citra sumber bersifat unik dan bebas duplikasi.


In [ ]:
import zipfile
from pathlib import Path

# Ekstraksi berkas ZIP dataset jika direktori belum ada
if not LOCAL_EXTRACT_DIR.exists():
    if not ZIP_DATASET_SOURCE.exists():
        raise FileNotFoundError(f"Berkas arsip ZIP tidak ditemukan: {ZIP_DATASET_SOURCE}")

    print(f"Mengekstrak {ZIP_DATASET_SOURCE.name} ke {LOCAL_EXTRACT_DIR}...")
    LOCAL_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_DATASET_SOURCE, "r") as z:
        bad_file = z.testzip()
        if bad_file:
            raise RuntimeError(f"Arsip ZIP mengalami kerusakan pada file: {bad_file}")
        z.extractall(LOCAL_EXTRACT_DIR)
    print("Ekstraksi selesai.")
else:
    print(f"Direktori {LOCAL_EXTRACT_DIR} sudah tersedia. Ekstraksi dilewati.")

# Verifikasi struktur subfolder
for split_name in ["train", "valid"]:
    img_folder = LOCAL_EXTRACT_DIR / split_name / "images"
    lbl_folder = LOCAL_EXTRACT_DIR / split_name / "labels"
    print(f"Dataset sumber [{split_name}]: {len(list(img_folder.glob('*')))} citra, {len(list(lbl_folder.glob('*.txt')))} label")


Mengekstrak yolov8-helmet-final-use-no-augmentation.v2-no-augmentasi.yolov8.zip ke /content/dataset_helmet...
Ekstraksi selesai.
Dataset sumber [train]: 722 citra, 722 label
Dataset sumber [valid]: 186 citra, 186 label


In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib

# Pemeriksaan keunikan citra lintas train dan valid menggunakan hash SHA-256
def hash_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

kelompok_hash = defaultdict(list)

for split in ["train", "valid"]:
    folder_gambar = LOCAL_EXTRACT_DIR / split / "images"
    for file in folder_gambar.iterdir():
        if file.is_file():
            kelompok_hash[hash_file(file)].append((split, file.name))

total_unik = len(kelompok_hash)
duplikat_lintas = [k for k, v in kelompok_hash.items() if len({split for split, name in v}) > 1]

print("Hasil Analisis Hash SHA-256 Citra Sumber:")
print(f"  - Total citra unik berdasarkan hash: {total_unik}")
print(f"  - Kelompok duplikat lintas train/valid: {len(duplikat_lintas)}")

if len(duplikat_lintas) > 0:
    raise ValueError(f"Ditemukan {len(duplikat_lintas)} citra identik antara folder train dan valid!")
else:
    print("Status: VALID (Seluruh citra sumber bebas kontaminasi duplikasi lintas split).")


Hasil Analisis Hash SHA-256 Citra Sumber:
  - Total citra unik berdasarkan hash: 908
  - Kelompok duplikat lintas train/valid: 0
Status: VALID (Seluruh citra sumber bebas kontaminasi duplikasi lintas split).


## Bagian 7: Pembagian Dataset Terstratifikasi (Train, Valid, Test)

Dataset sumber (908 citra) digabungkan dan dibagi ulang dengan rasio **70% : 15% : 15%** menggunakan `random.seed(42)` untuk memastikan keterulangan (*reproducibility*).

Strategi pembagian:
- **Train (636 citra)**
- **Valid (137 citra)**
- **Test (135 citra)**
- Diterapkan stratifikasi berdasarkan kemunculan kombinasi kelas anotasi pada tiap citra.
- Menggunakan *safety guard* agar tidak menimpa dataset yang sudah dibagi bila sel dijalankan ulang.


In [ ]:
from pathlib import Path
from collections import Counter
import random
import shutil

SOURCE = LOCAL_EXTRACT_DIR
DEST = LOCAL_SPLIT_DIR
SEED = 42

random.seed(SEED)

if DEST.exists():
    print(f"Direktori {DEST} sudah ada.")
    print("Pembagian ulang dilewati untuk mencegah perubahan data. Hapus folder tersebut jika ingin membuat ulang.")
else:
    # 1. Kumpulkan seluruh pasangan citra dan label
    records = []
    for split in ["train", "valid"]:
        img_dir = SOURCE / split / "images"
        lbl_dir = SOURCE / split / "labels"

        for img in sorted(img_dir.iterdir()):
            if not img.is_file():
                continue
            label = lbl_dir / f"{img.stem}.txt"
            if not label.exists():
                raise FileNotFoundError(f"Label tidak ditemukan untuk citra: {img.name}")
            records.append((img, label))

    # Pastikan tidak ada duplikasi nama file stem
    stems = [img.stem for img, _ in records]
    if len(stems) != len(set(stems)):
        raise ValueError("Terdapat nama file yang bertabrakan sebelum pembagian!")

    # 2. Fungsi membaca kelas anotasi per citra untuk stratifikasi
    def get_classes(label_path):
        classes = set()
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if parts:
                classes.add(int(float(parts[0])))
        return classes

    # Pengelompokan stratifikasi berdasarkan kombinasi kelas
    groups = {}
    for record in records:
        classes = tuple(sorted(get_classes(record[1])))
        groups.setdefault(classes, []).append(record)

    splits = {"train": [], "valid": [], "test": []}

    for group in groups.values():
        random.shuffle(group)
        n = len(group)
        n_train = round(n * 0.70)
        n_valid = round(n * 0.15)

        splits["train"].extend(group[:n_train])
        splits["valid"].extend(group[n_train:n_train + n_valid])
        splits["test"].extend(group[n_train + n_valid:])

    # 3. Salin file ke struktur folder terpisah
    for split_name, items in splits.items():
        for kind in ["images", "labels"]:
            (DEST / split_name / kind).mkdir(parents=True, exist_ok=True)
        for img, label in items:
            shutil.copy2(img, DEST / split_name / "images" / img.name)
            shutil.copy2(label, DEST / split_name / "labels" / label.name)

    print("Pembagian dataset terstratifikasi berhasil diselesaikan.")
    for split_name, items in splits.items():
        print(f"  - {split_name}: {len(items)} citra")
    print("Total:", sum(len(v) for v in splits.values()), "citra")


Pembagian dataset terstratifikasi berhasil diselesaikan.
  - train: 636 citra
  - valid: 137 citra
  - test: 135 citra
Total: 908 citra


In [ ]:
from pathlib import Path
from collections import Counter

# Verifikasi kuantitatif hasil pembagian dataset dan distribusi kelas objek
ROOT = LOCAL_SPLIT_DIR
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

total_gambar = 0
total_label = 0
semua_lolos = True

for split in ["train", "valid", "test"]:
    img_dir = ROOT / split / "images"
    lbl_dir = ROOT / split / "labels"

    images = [p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in EXTS]
    labels = list(lbl_dir.glob("*.txt"))

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}

    missing_labels = image_stems - label_stems
    orphan_labels = label_stems - image_stems

    class_counts = Counter()
    empty_labels = 0
    invalid_lines = 0

    for label_file in labels:
        lines = label_file.read_text().splitlines()
        if not lines:
            empty_labels += 1
            continue
        for line in lines:
            parts = line.split()
            if len(parts) == 5:
                try:
                    cls_id = int(parts[0])
                    class_counts[cls_id] += 1
                except ValueError:
                    invalid_lines += 1
            else:
                invalid_lines += 1

    ok = (
        len(missing_labels) == 0
        and len(orphan_labels) == 0
        and empty_labels == 0
        and invalid_lines == 0
    )
    semua_lolos &= ok

    print(f"--- {split.upper()} ---")
    print(f"Gambar: {len(images)}")
    print(f"Label: {len(labels)}")
    print(f"Label hilang: {len(missing_labels)}")
    print(f"Label tanpa gambar: {len(orphan_labels)}")
    print(f"Jumlah objek class 0 (helmet): {class_counts[0]}")
    print(f"Jumlah objek class 1 (no-helmet): {class_counts[1]}")
    print(f"Baris label kosong: {empty_labels}")
    print(f"Baris label tidak valid: {invalid_lines}")
    print(f"Status: {'LOLOS' if ok else 'PERIKSA'}\n")

    total_gambar += len(images)
    total_label += len(labels)

print(f"Total gambar: {total_gambar}")
print(f"Total file label: {total_label}")
print("HASIL AKHIR:", "SEMUA LOLOS" if semua_lolos else "ADA MASALAH")


--- TRAIN ---
Gambar: 636
Label: 636
Label hilang: 0
Label tanpa gambar: 0
Jumlah objek class 0 (helmet): 1413
Jumlah objek class 1 (no-helmet): 902
Baris label kosong: 0
Baris label tidak valid: 0
Status: LOLOS

--- VALID ---
Gambar: 137
Label: 137
Label hilang: 0
Label tanpa gambar: 0
Jumlah objek class 0 (helmet): 320
Jumlah objek class 1 (no-helmet): 195
Baris label kosong: 0
Baris label tidak valid: 0
Status: LOLOS

--- TEST ---
Gambar: 135
Label: 135
Label hilang: 0
Label tanpa gambar: 0
Jumlah objek class 0 (helmet): 291
Jumlah objek class 1 (no-helmet): 188
Baris label kosong: 0
Baris label tidak valid: 0
Status: LOLOS

Total gambar: 908
Total file label: 908
HASIL AKHIR: SEMUA LOLOS


In [ ]:
from pathlib import Path

# Verifikasi kepatuhan nilai koordinat bounding box YOLO (0 <= val <= 1, w > 0, h > 0)
ROOT = LOCAL_SPLIT_DIR
jumlah_error = 0

for split in ["train", "valid", "test"]:
    folder = ROOT / split / "labels"
    error_split = 0

    for file in folder.glob("*.txt"):
        for nomor, baris in enumerate(file.read_text().splitlines(), start=1):
            bagian = baris.split()
            try:
                if len(bagian) != 5:
                    raise ValueError("Jumlah kolom bukan 5")
                kelas = int(bagian[0])
                x, y, w, h = map(float, bagian[1:])
                if kelas not in [0, 1]:
                    raise ValueError("ID kelas tidak dikenal")
                if not all(0 <= v <= 1 for v in [x, y, w, h]):
                    raise ValueError("Koordinat di luar rentang 0-1")
                if w <= 0 or h <= 0:
                    raise ValueError("Lebar atau tinggi tidak valid")
            except Exception as e:
                error_split += 1
                jumlah_error += 1
                if error_split <= 5:
                    print(f"{split}/{file.name}, baris {nomor}: {e}")

    print(f"{split}: {error_split} baris bermasalah")

print()
print("Total baris bermasalah:", jumlah_error)
print("Status Integritas Label: VALID (Semua koordinat berada pada rentang batas yang benar).")

train: 0 baris bermasalah
valid: 0 baris bermasalah
test: 0 baris bermasalah

Total baris bermasalah: 0
Status Integritas Label: VALID (Semua koordinat berada pada rentang batas yang benar).


## Bagian 8: Pembuatan Folder Dataset Dasar Skenario M0–M4 dan Berkas data.yaml

Pada tahap ini, folder kerja untuk kelima skenario (`M0`, `M1`, `M2`, `M3`, dan `M4`) dibuat di bawah `/content/eksperimen_final_M0_M4/`. Seluruh skenario pada awalnya memuat citra dasar yang sama persis (636 train, 137 valid, 135 test).

Setiap skenario dilengkapi berkas konfigurasi `data.yaml` resmi yang mendefinisikan:
- `path`: Direktori absolut skenario
- `train`: `train/images`
- `val`: `valid/images`
- `test`: `test/images`
- `nc`: 2
- `names`: `['helmet', 'no-helmet']`


In [ ]:
from pathlib import Path
import shutil

SOURCE = LOCAL_SPLIT_DIR
DEST = LOCAL_EXPERIMENT_ROOT

if DEST.exists():
    print(f"Folder {DEST} sudah ada. Struktur dasar dipertahankan (tidak ada file yang ditimpa).")
else:
    for scenario in ["M0", "M1", "M2", "M3", "M4"]:
        for split in ["train", "valid", "test"]:
            for kind in ["images", "labels"]:
                src = SOURCE / split / kind
                dst = DEST / scenario / split / kind
                dst.mkdir(parents=True, exist_ok=True)
                for file in src.iterdir():
                    if file.is_file():
                        shutil.copy2(file, dst / file.name)
        print(f"{scenario}: struktur dasar berhasil disiapkan")

    print(f"\nFolder eksperimen: {DEST}")


M0: struktur dasar berhasil disiapkan
M1: struktur dasar berhasil disiapkan
M2: struktur dasar berhasil disiapkan
M3: struktur dasar berhasil disiapkan
M4: struktur dasar berhasil disiapkan

Folder eksperimen: /content/eksperimen_final_M0_M4


In [ ]:
from pathlib import Path
import yaml

ROOT = LOCAL_EXPERIMENT_ROOT

# Pembuatan data.yaml resmi untuk setiap skenario M0 s.d. M4
for scenario in ["M0", "M1", "M2", "M3", "M4"]:
    folder = ROOT / scenario
    config = {
        "path": str(folder),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": 2,
        "names": ["helmet", "no-helmet"]
    }
    yaml_path = folder / "data.yaml"
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

print("Berkas data.yaml untuk seluruh skenario berhasil diperiksa dan dibuat.")


Berkas data.yaml untuk seluruh skenario berhasil diperiksa dan dibuat.


In [ ]:
from pathlib import Path
import yaml

ROOT = LOCAL_EXPERIMENT_ROOT

# Verifikasi keberadaan folder yang dirujuk oleh data.yaml
for scenario in ["M0", "M1", "M2", "M3", "M4"]:
    config_path = ROOT / scenario / "data.yaml"
    with open(config_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    base = Path(cfg["path"])
    print(f"\n{scenario}")
    for key in ["train", "val", "test"]:
        resolved = base / cfg[key]
        print(f"  {key}: {'ADA' if resolved.is_dir() else 'TIDAK ADA'}")
    print("  Kelas:", cfg["names"])



M0
  train: ADA
  val: ADA
  test: ADA
  Kelas: ['helmet', 'no-helmet']

M1
  train: ADA
  val: ADA
  test: ADA
  Kelas: ['helmet', 'no-helmet']

M2
  train: ADA
  val: ADA
  test: ADA
  Kelas: ['helmet', 'no-helmet']

M3
  train: ADA
  val: ADA
  test: ADA
  Kelas: ['helmet', 'no-helmet']

M4
  train: ADA
  val: ADA
  test: ADA
  Kelas: ['helmet', 'no-helmet']


## Bagian 9: Pembuatan Augmentasi Offline (M1, M2, dan M3)

Augmentasi offline hanya diterapkan pada subset **`train`** skenario M1, M2, dan M3:
- **M1 (Horizontal Flip)**: Citra dibalik horizontal (`cv2.flip(img, 1)`), koordinat bounding box $x$ ditransformasi menjadi $1.0 - x$.
- **M2 (Rotasi $\pm 5^\circ$)**: Citra dirotasi bergantian $+5^\circ$ dan $-5^\circ$. Titik-titik sudut bounding box ditransformasi secara affine dan dipotong (*clipped*) agar tidak melebihi dimensi citra. Log sudut rotasi dan validitas anotasi disimpan ke `sudut_rotasi.csv`.
- **M3 (Brightness & Contrast)**: Dilakukan penyesuaian intensitas piksel dengan skala linear $lpha=1.15$ dan offset $eta=15$ (`cv2.convertScaleAbs`), label anotasi tetap sama.

> **Pencegahan Augmentasi Ganda:**
> Kode memeriksa keberadaan file `*_aug.*`. Jika sudah ada, pembuatan augmentasi dilewati untuk mencegah penggandaan data yang tidak diinginkan.


In [ ]:
from pathlib import Path
import cv2
import numpy as np
import random
import csv

ROOT = LOCAL_EXPERIMENT_ROOT
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def read_yolo_label(path):
    rows = []
    for line in path.read_text().splitlines():
        parts = line.split()
        if parts:
            if len(parts) != 5:
                raise ValueError(f"Label tidak valid: {path}")
            rows.append([int(parts[0]), *map(float, parts[1:])])
    return rows

def write_yolo_label(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                f"{int(row[0])} "
                f"{row[1]:.6f} {row[2]:.6f} "
                f"{row[3]:.6f} {row[4]:.6f}\n"
            )

def save_image(path, image):
    if not cv2.imwrite(str(path), image):
        raise RuntimeError(f"Gagal menyimpan gambar: {path}")

# Periksa apakah augmentasi offline sudah pernah diterapkan
sudah_ada_aug = False
for scenario in ["M1", "M2", "M3"]:
    image_dir = ROOT / scenario / "train" / "images"
    existing = list(image_dir.glob("*_aug.*"))
    if existing:
        sudah_ada_aug = True
        print(f"[{scenario}] Augmentasi offline sudah ada ({len(existing)} citra). Pembuatan ulang dilewati.")

if not sudah_ada_aug:
    logs_rotation = []
    counts = {}

    for scenario in ["M1", "M2", "M3"]:
        image_dir = ROOT / scenario / "train" / "images"
        label_dir = ROOT / scenario / "train" / "labels"

        originals = [
            p for p in sorted(image_dir.iterdir())
            if p.is_file() and p.suffix.lower() in EXTS and "_aug" not in p.stem
        ]

        created = 0

        for idx, image_path in enumerate(originals):
            label_path = label_dir / f"{image_path.stem}.txt"
            if not label_path.exists():
                raise FileNotFoundError(label_path)

            image = cv2.imread(str(image_path))
            if image is None:
                raise RuntimeError(f"Gagal membaca: {image_path}")

            h, w = image.shape[:2]
            labels = read_yolo_label(label_path)
            aug_image = None
            aug_labels = []

            if scenario == "M1":
                # Horizontal Flip
                aug_image = cv2.flip(image, 1)
                aug_labels = [
                    [cls, 1.0 - x, y, bw, bh]
                    for cls, x, y, bw, bh in labels
                ]

            elif scenario == "M2":
                # Rotasi terkontrol +/- 5 derajat
                angle = 5 if idx % 2 == 0 else -5
                matrix = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
                aug_image = cv2.warpAffine(
                    image, matrix, (w, h),
                    flags=cv2.INTER_LINEAR,
                    borderMode=cv2.BORDER_CONSTANT,
                    borderValue=(114, 114, 114)
                )

                for cls, x, y, bw, bh in labels:
                    cx, cy = x * w, y * h
                    bw_px, bh_px = bw * w, bh * h

                    corners = np.array([
                        [cx - bw_px/2, cy - bh_px/2, 1],
                        [cx + bw_px/2, cy - bh_px/2, 1],
                        [cx + bw_px/2, cy + bh_px/2, 1],
                        [cx - bw_px/2, cy + bh_px/2, 1]
                    ])

                    rotated = corners @ matrix.T
                    x1 = np.clip(rotated[:, 0].min(), 0, w)
                    y1 = np.clip(rotated[:, 1].min(), 0, h)
                    x2 = np.clip(rotated[:, 0].max(), 0, w)
                    y2 = np.clip(rotated[:, 1].max(), 0, h)

                    if x2 <= x1 or y2 <= y1:
                        continue

                    aug_labels.append([
                        cls,
                        ((x1 + x2) / 2) / w,
                        ((y1 + y2) / 2) / h,
                        (x2 - x1) / w,
                        (y2 - y1) / h
                    ])

                logs_rotation.append([
                    image_path.name, angle,
                    len(labels), len(aug_labels)
                ])

            elif scenario == "M3":
                # Brightness dan Contrast
                alpha = 1.15
                beta = 15
                aug_image = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
                aug_labels = labels.copy()

            output_image = image_dir / f"{image_path.stem}_aug.jpg"
            output_label = label_dir / f"{image_path.stem}_aug.txt"

            save_image(output_image, aug_image)
            write_yolo_label(output_label, aug_labels)
            created += 1

        counts[scenario] = created
        print(f"{scenario}: {created} gambar augmentasi dibuat")

    # Simpan catatan rotasi M2
    log_path = ROOT / "M2" / "sudut_rotasi.csv"
    with open(log_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["nama_gambar", "sudut_derajat", "jumlah_label_asli", "jumlah_label_hasil"])
        writer.writerows(logs_rotation)

    print("\nJumlah augmentasi dibuat:", counts)
    print("Log rotasi tersimpan di:", log_path)


M1: 636 gambar augmentasi dibuat
M2: 636 gambar augmentasi dibuat
M3: 636 gambar augmentasi dibuat

Jumlah augmentasi dibuat: {'M1': 636, 'M2': 636, 'M3': 636}
Log rotasi tersimpan di: /content/eksperimen_final_M0_M4/M2/sudut_rotasi.csv


## Bagian 10: Spesifikasi Parameter Augmentasi Online M4 dan Isolasi M0–M3

Untuk menjaga kemurnian eksperimen, parameter augmentasi diatur secara terisolasi:
- **M0, M1, M2, M3**: Seluruh augmentasi online Ultralytics dimatikan murni ke nilai `0.0` (`mosaic`, `mixup`, `copy_paste`, `degrees`, `translate`, `scale`, `shear`, `perspective`, `fliplr`, `flipud`, `bgr`, `hsv_h`, `hsv_s`, `hsv_v`, `erasing`, `auto_augment=None`).
- **M4**: Menggunakan parameter augmentasi online bawaan resmi YOLOv8 yang didefinisikan secara eksplisit.


In [ ]:
# ==============================================================================
# Spesifikasi Parameter Augmentasi Online Skenario M0 s.d. M4
# ==============================================================================

# Parameter augmentasi online dinonaktifkan murni (untuk M0, M1, M2, M3)
NO_ONLINE_AUG = {
    "mosaic": 0.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "degrees": 0.0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "fliplr": 0.0,
    "flipud": 0.0,
    "bgr": 0.0,
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.0,
    "erasing": 0.0,
    "crop_fraction": 1.0,
    "auto_augment": None
}

# Parameter augmentasi online bawaan YOLOv8 ditentukan secara eksplisit (khusus M4)
M4_ONLINE_AUG = {
    "mosaic": 1.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "fliplr": 0.5,
    "flipud": 0.0,
    "bgr": 0.0,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "erasing": 0.4,
    "crop_fraction": 1.0,
    "auto_augment": "randaugment"
}

SCENARIO_AUG_CONFIGS = {
    "M0": {**NO_ONLINE_AUG, "keterangan": "Baseline murni (tanpa offline & tanpa online aug)"},
    "M1": {**NO_ONLINE_AUG, "keterangan": "Horizontal flip offline saja (online aug dimatikan)"},
    "M2": {**NO_ONLINE_AUG, "keterangan": "Rotasi +/- 5 deg offline saja (online aug dimatikan)"},
    "M3": {**NO_ONLINE_AUG, "keterangan": "Brightness & contrast offline saja (online aug dimatikan)"},
    "M4": {**M4_ONLINE_AUG, "keterangan": "Ultralytics default online aug pada dataset dasar"}
}

print("Konfigurasi parameter augmentasi untuk 5 skenario siap:")
for scn, cfg in SCENARIO_AUG_CONFIGS.items():
    print(f"  [{scn}] {cfg['keterangan']}")


Konfigurasi parameter augmentasi untuk 5 skenario siap:
  [M0] Baseline murni (tanpa offline & tanpa online aug)
  [M1] Horizontal flip offline saja (online aug dimatikan)
  [M2] Rotasi +/- 5 deg offline saja (online aug dimatikan)
  [M3] Brightness & contrast offline saja (online aug dimatikan)
  [M4] Ultralytics default online aug pada dataset dasar


## Bagian 11: Validasi Komprehensif Seluruh Skenario (M0–M4)

Rangkaian pengujian integritas dilakukan sebelum masuk ke tahap training:
1. **Validasi Bounding Box dan Pasangan Citra-Label**: Memastikan tidak ada label yang hilang, tidak ada label tanpa citra, dan seluruh koordinat valid.
2. **Uji Konsistensi Hash SHA-256 Valid & Test**: Memastikan data uji (*validation* dan *test*) identik 100% di semua skenario.
3. **Pemeriksaan Kualitas Rotasi M2**: Menjamin rotasi $\pm 5^\circ$ tidak menyebabkan label kosong (*empty label*).
4. **Verifikasi Ultralytics Compatibility**: Memastikan format `data.yaml` dan folder siap untuk eksekusi training.


In [ ]:
from pathlib import Path

ROOT = LOCAL_EXPERIMENT_ROOT
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

semua_lolos = True

for scenario in ["M0", "M1", "M2", "M3", "M4"]:
    print(f"\n--- {scenario} ---")
    for split in ["train", "valid", "test"]:
        img_dir = ROOT / scenario / split / "images"
        lbl_dir = ROOT / scenario / split / "labels"

        images = [p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in EXTS]
        labels = list(lbl_dir.glob("*.txt"))

        image_stems = {p.stem for p in images}
        label_stems = {p.stem for p in labels}
        missing = image_stems - label_stems
        extra = label_stems - image_stems

        bad_boxes = 0
        for label in labels:
            for line in label.read_text().splitlines():
                parts = line.split()
                try:
                    if len(parts) != 5:
                        raise ValueError()
                    cls = int(parts[0])
                    x, y, w, h = map(float, parts[1:])
                    if (
                        cls not in [0, 1]
                        or not all(0 <= v <= 1 for v in [x, y, w, h])
                        or w <= 0 or h <= 0
                    ):
                        raise ValueError()
                except Exception:
                    bad_boxes += 1

        ok = not missing and not extra and bad_boxes == 0
        semua_lolos &= ok

        print(
            f"{split}: gambar={len(images)}, label={len(labels)}, "
            f"label_hilang={len(missing)}, "
            f"label_tanpa_gambar={len(extra)}, "
            f"baris_bbox_bermasalah={bad_boxes}, "
            f"status={'LOLOS' if ok else 'PERIKSA'}"
        )

print("\nHASIL AKHIR VALIDASI:", "SEMUA LOLOS" if semua_lolos else "PERLU DIPERIKSA")



--- M0 ---
train: gambar=636, label=636, label_hilang=0, label_tanpa_gambar=0, baris_bbox_bermasalah=0, status=LOLOS
valid: gambar=137, label=137, label_hilang=0, label_tanpa_gambar=0, baris_bbox_bermasalah=0, status=LOLOS
test: gambar=135, label=135, label_hilang=0, label_tanpa_gambar=0, baris_bbox_bermasalah=0, status=LOLOS

--- M1 ---
train: gambar=1272, label=1272, label_hilang=0, label_tanpa_gambar=0, baris_bbox_bermasalah=0, status=LOLOS
valid: gambar=137, label=137, label_hilang=0, label_tanpa_gambar=0, baris_bbox_bermasalah=0, status=LOLOS
test: gambar=135, label=135, label_hilang=0, label_tanpa_gambar=0, baris_bbox_bermasalah=0, status=LOLOS

--- M2 ---
train: gambar=1272, label=1272, label_hilang=0, label_tanpa_gambar=0, baris_bbox_bermasalah=0, status=LOLOS
valid: gambar=137, label=137, label_hilang=0, label_tanpa_gambar=0, baris_bbox_bermasalah=0, status=LOLOS
test: gambar=135, label=135, label_hilang=0, label_tanpa_gambar=0, baris_bbox_bermasalah=0, status=LOLOS

--- M3 -

In [ ]:
from pathlib import Path
import hashlib

ROOT = LOCAL_EXPERIMENT_ROOT

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

semua_sama = True

for split in ["valid", "test"]:
    print(f"\n--- {split.upper()} ---")
    baseline = ROOT / "M0" / split

    for scenario in ["M1", "M2", "M3", "M4"]:
        sama = True
        for kind in ["images", "labels"]:
            base_dir = baseline / kind
            other_dir = ROOT / scenario / split / kind

            base_files = {p.name: sha256(p) for p in base_dir.iterdir() if p.is_file()}
            other_files = {p.name: sha256(p) for p in other_dir.iterdir() if p.is_file()}

            if base_files != other_files:
                sama = False

        print(f"M0 vs {scenario}: {'IDENTIK' if sama else 'BERBEDA'}")
        semua_sama &= sama

print("\nHASIL UJI EVALUASI LINTAS SKENARIO:", "SEMUA IDENTIK" if semua_sama else "ADA PERBEDAAN")



--- VALID ---
M0 vs M1: IDENTIK
M0 vs M2: IDENTIK
M0 vs M3: IDENTIK
M0 vs M4: IDENTIK

--- TEST ---
M0 vs M1: IDENTIK
M0 vs M2: IDENTIK
M0 vs M3: IDENTIK
M0 vs M4: IDENTIK

HASIL UJI EVALUASI LINTAS SKENARIO: SEMUA IDENTIK


In [ ]:
from pathlib import Path
import csv

ROOT = LOCAL_EXPERIMENT_ROOT

# Periksa label kosong pada gambar hasil rotasi M2
img_dir = ROOT / "M2" / "train" / "images"
lbl_dir = ROOT / "M2" / "train" / "labels"

aug_images = [p for p in img_dir.iterdir() if p.is_file() and p.stem.endswith("_aug")]
empty_labels = []
for img in aug_images:
    label = lbl_dir / f"{img.stem}.txt"
    if label.exists() and not label.read_text().strip():
        empty_labels.append(img.name)

print("Status Augmentasi Rotasi M2:")
print(f"  - Jumlah citra augmentasi rotasi: {len(aug_images)}")
print(f"  - Citra rotasi dengan label kosong: {len(empty_labels)}")

# Periksa log rotasi
log_file = ROOT / "M2" / "sudut_rotasi.csv"
if log_file.exists():
    with open(log_file, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    print(f"  - Total baris log tercatat: {len(rows)}")
    print("  - Contoh entri log pertama:")
    for row in rows[:3]:
        print("   ", row)
else:
    print("  - Berkas log sudut_rotasi.csv belum ditemukan.")


Status Augmentasi Rotasi M2:
  - Jumlah citra augmentasi rotasi: 636
  - Citra rotasi dengan label kosong: 0
  - Total baris log tercatat: 636
  - Contoh entri log pertama:
    {'nama_gambar': '03_png.rf.1852a93481be649c94894892db0669c1.jpg', 'sudut_derajat': '5', 'jumlah_label_asli': '2', 'jumlah_label_hasil': '2'}
    {'nama_gambar': '1002_png.rf.c6d1144f05f2fbbc60a889516d37d884.jpg', 'sudut_derajat': '-5', 'jumlah_label_asli': '2', 'jumlah_label_hasil': '2'}
    {'nama_gambar': '1003_png.rf.9aa0e04c3bee5b8c299da1e4906b335a.jpg', 'sudut_derajat': '5', 'jumlah_label_asli': '1', 'jumlah_label_hasil': '1'}


In [ ]:
from pathlib import Path
import yaml
import ultralytics

ROOT = LOCAL_EXPERIMENT_ROOT
SCENARIOS = ["M0", "M1", "M2", "M3", "M4"]

print("Ultralytics Version:", ultralytics.__version__)
print("Status Folder Eksperimen:", ROOT.exists())

semua_lolos = True

for scenario in SCENARIOS:
    folder = ROOT / scenario
    yaml_file = folder / "data.yaml"
    print(f"\n--- {scenario} ---")
    if not folder.exists() or not yaml_file.exists():
        print("PERIKSA: Folder atau data.yaml tidak ditemukan")
        semua_lolos = False
        continue

    with open(yaml_file, encoding="utf-8") as f:
        config = yaml.safe_load(f)

    if config.get("nc") != 2 or config.get("names") != ["helmet", "no-helmet"]:
        print("PERIKSA: Konfigurasi kelas tidak sesuai")
        semua_lolos = False

    for split in ["train", "valid", "test"]:
        images = folder / split / "images"
        labels = folder / split / "labels"
        if not images.is_dir() or not labels.is_dir():
            print(f"PERIKSA: Folder {split} tidak lengkap")
            semua_lolos = False
        else:
            print(f"{split}: folder tersedia")

print("\nHASIL PEMERIKSAAN KESIAPAN YOLOV8:", "LOLOS" if semua_lolos else "PERLU DIPERIKSA")


Ultralytics Version: 8.4.182
Status Folder Eksperimen: True

--- M0 ---
train: folder tersedia
valid: folder tersedia
test: folder tersedia

--- M1 ---
train: folder tersedia
valid: folder tersedia
test: folder tersedia

--- M2 ---
train: folder tersedia
valid: folder tersedia
test: folder tersedia

--- M3 ---
train: folder tersedia
valid: folder tersedia
test: folder tersedia

--- M4 ---
train: folder tersedia
valid: folder tersedia
test: folder tersedia

HASIL PEMERIKSAAN KESIAPAN YOLOV8: LOLOS


## Bagian 12: Pemeriksaan Akselerator GPU dan Konfigurasi Hyperparameter

Bagian ini memvalidasi ketersediaan GPU NVIDIA Tesla T4 dan menetapkan konfigurasi hyperparameter global yang seragam di seluruh skenario training.


In [ ]:
import torch

print("Status Akselerator Hardware:")
print("  - CUDA Tersedia :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("  - Nama Perangkat:", torch.cuda.get_device_name(0))
    print("  - Versi CUDA    :", torch.version.cuda)
    print("  - Memori GPU    :", f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("PERINGATAN: GPU belum terdeteksi. Silakan aktifkan via Runtime > Change runtime type > T4 GPU.")


Status Akselerator Hardware:
  - CUDA Tersedia : True
  - Nama Perangkat: Tesla T4
  - Versi CUDA    : 13.0
  - Memori GPU    : 14.56 GB


## Bagian 13: Konfigurasi Output Persisten Google Drive

Seluruh artefak hasil training (bobot terbaik `best.pt`, kurva loss, metrik, confusion matrix) dan hasil evaluasi disimpan secara terstruktur ke Google Drive agar aman dari terminasi runtime Colab:
- Folder Training: `/content/drive/MyDrive/Penelitian_YOLOv8/hasil_eksperimen_final_M0_M4/training_runs/{scenario}/`
- Folder Evaluasi: `/content/drive/MyDrive/Penelitian_YOLOv8/hasil_eksperimen_final_M0_M4/testing_evals/{scenario}/`


In [ ]:
from pathlib import Path
import os

# Direktori penyimpanan permanen hasil di Google Drive
EXPERIMENT_OUTPUT = Path("/content/drive/MyDrive/Penelitian_YOLOv8/hasil_eksperimen_final_M0_M4")
TRAIN_RUNS_DIR = EXPERIMENT_OUTPUT / "training_runs"
TEST_EVALS_DIR = EXPERIMENT_OUTPUT / "testing_evals"

TRAIN_RUNS_DIR.mkdir(parents=True, exist_ok=True)
TEST_EVALS_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameter Konsisten (YOLOv8n, epochs=50, imgsz=640, batch=16, seed=42)
GLOBAL_CONFIG = {
    "model_base": "yolov8n.pt",
    "epochs": 50,
    "patience": 15,
    "imgsz": 640,
    "batch": 16,
    "device": 0,
    "workers": 2,
    "seed": 42,
    "deterministic": True,
    "optimizer": "auto",
    "pretrained": True,
    "val": True,
    "save": True,
    "verbose": True
}

print("Konfigurasi Penyimpanan Persisten:")
print("  - Direktori Output Training :", TRAIN_RUNS_DIR)
print("  - Direktori Output Evaluasi :", TEST_EVALS_DIR)
print("\nHyperparameter Global Training:")
for k, v in GLOBAL_CONFIG.items():
    print(f"  - {k:<15}: {v}")


Konfigurasi Penyimpanan Persisten:
  - Direktori Output Training : /content/drive/MyDrive/Penelitian_YOLOv8/hasil_eksperimen_final_M0_M4/training_runs
  - Direktori Output Evaluasi : /content/drive/MyDrive/Penelitian_YOLOv8/hasil_eksperimen_final_M0_M4/testing_evals

Hyperparameter Global Training:
  - model_base     : yolov8n.pt
  - epochs         : 50
  - patience       : 15
  - imgsz          : 640
  - batch          : 16
  - device         : 0
  - workers        : 2
  - seed           : 42
  - deterministic  : True
  - optimizer      : auto
  - pretrained     : True
  - val            : True
  - save           : True
  - verbose        : True


## Bagian 14: Eksekusi Training YOLOv8 (M0 s.d. M4) — Belum Dijalankan

> [!IMPORTANT]
> **STATUS: BELUM DIJALANKAN.**
> Sel di bawah ini siap dieksekusi saat Anda memulai sesi training di Google Colab.
>
> **Fitur Keamanan:**
> - Setiap skenario memuat ulang model segar dari `yolov8n.pt` (*pretrained=True*) untuk mencegah *weight leakage*.
> - Dilengkapi validasi ganda `training_completed.flag` dan `best.pt.exists()`. Skenario yang terputus di tengah jalan akan diulang dari awal secara bersih, bukan dilewati.
> - Log durasi dan status training diperbarui secara inkremental ke `training_summary_log.json` di Google Drive setiap kali satu skenario selesai.


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import time
import json

ROOT_DATASET = Path("/content/eksperimen_final_M0_M4")
summary_log_file = EXPERIMENT_OUTPUT / "training_summary_log.json"

# Muat log yang sudah ada sebelumnya jika ada
if summary_log_file.exists():
    try:
        with open(summary_log_file, "r", encoding="utf-8") as f:
            training_log_summary = json.load(f)
    except Exception:
        training_log_summary = {}
else:
    training_log_summary = {}

for scenario in ["M0", "M1", "M2", "M3", "M4"]:
    print(f"\n{'='*35} SKENARIO: {scenario} {'='*35}")

    yaml_path = ROOT_DATASET / scenario / "data.yaml"
    if not yaml_path.exists():
        raise FileNotFoundError(f"data.yaml untuk {scenario} tidak ditemukan: {yaml_path}")

    target_scenario_dir = TRAIN_RUNS_DIR / scenario
    best_model_path = target_scenario_dir / "weights" / "best.pt"
    completion_flag = target_scenario_dir / "training_completed.flag"

    # Mekanisme skip aman: Hanya lewati jika training terbukti telah tuntas
    if completion_flag.exists() and best_model_path.exists():
        print(f"[{scenario}] Training telah tuntas sebelumnya (flag terdeteksi). Melewati skenario ini.")
        if scenario not in training_log_summary:
            training_log_summary[scenario] = {
                "status": "SUDAH_TUNTAS",
                "best_weight": str(best_model_path)
            }
        continue
    elif best_model_path.exists() and not completion_flag.exists():
        print(f"[{scenario}] PERINGATAN: best.pt ditemukan namun training_completed.flag tidak ada (sesi terputus). Training akan diulang secara bersih.")

    # Muat ulang model awal segar dari yolov8n.pt (pretrained=True)
    model = YOLO(GLOBAL_CONFIG["model_base"])

    # Gabungkan hyperparameter global dan konfigurasi augmentasi skenario
    train_args = {
        "data": str(yaml_path),
        "epochs": GLOBAL_CONFIG["epochs"],
        "patience": GLOBAL_CONFIG["patience"],
        "imgsz": GLOBAL_CONFIG["imgsz"],
        "batch": GLOBAL_CONFIG["batch"],
        "device": GLOBAL_CONFIG["device"],
        "workers": GLOBAL_CONFIG["workers"],
        "seed": GLOBAL_CONFIG["seed"],
        "deterministic": GLOBAL_CONFIG["deterministic"],
        "optimizer": GLOBAL_CONFIG["optimizer"],
        "pretrained": GLOBAL_CONFIG["pretrained"],
        "val": GLOBAL_CONFIG["val"],
        "save": GLOBAL_CONFIG["save"],
        "verbose": GLOBAL_CONFIG["verbose"],
        "project": str(TRAIN_RUNS_DIR),
        "name": scenario,
        "exist_ok": True,
        **{k: v for k, v in SCENARIO_AUG_CONFIGS[scenario].items() if k != "keterangan"}
    }

    start_time = time.time()
    results = model.train(**train_args)
    duration_min = (time.time() - start_time) / 60

    # Buat flag penanda tuntas
    completion_flag.write_text(
        f"SELESAI pada {time.strftime('%Y-%m-%d %H:%M:%S')}\nDurasi: {duration_min:.2f} menit",
        encoding="utf-8"
    )

    # Catat log dan simpan langsung ke Google Drive secara inkremental
    training_log_summary[scenario] = {
        "status": "SELESAI",
        "durasi_menit": round(duration_min, 2),
        "best_weight": str(best_model_path)
    }
    with open(summary_log_file, "w", encoding="utf-8") as f:
        json.dump(training_log_summary, f, indent=2)

    print(f"\nSelesai {scenario} dalam {duration_min:.2f} menit. Model terbaik: {best_model_path}")

print(f"\nSeluruh proses training selesai. Log persisten tersimpan di: {summary_log_file}")



=================================== SKENARIO: M0 ===================================
WARNING ⚠️ 'crop_fraction' is deprecated and will be removed in the future.
Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/eksperimen_final_M0_M4/M0/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.0, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio

## Bagian 15: Evaluasi Model Terbaik pada Split Test Independen — Belum Dijalankan

> [!IMPORTANT]
> **STATUS: BELUM DIJALANKAN.**
> Sel ini mengevaluasi bobot `best.pt` masing-masing skenario pada subset **Test (135 citra)** yang tidak pernah dilihat model selama training.
>
> **Metrik yang Dikumpulkan:**
> - Metrik Global: *Mean Precision* (`mp`), *Mean Recall* (`mr`), *mAP@0.50*, dan *mAP@0.50:0.95*.
> - Metrik Per Kelas: Precision, Recall, mAP50, dan mAP50-95 untuk `helmet` dan `no-helmet` menggunakan metode resmi `.class_result()`.
> - Visualisasi Evaluasi: Matriks konfusi (`confusion_matrix.png`) dan kurva PR yang disimpan permanen ke Google Drive.


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

ROOT_DATASET = Path("/content/eksperimen_final_M0_M4")
eval_records = []
per_class_records = []

for scenario in ["M0", "M1", "M2", "M3", "M4"]:
    best_weight = TRAIN_RUNS_DIR / scenario / "weights" / "best.pt"
    yaml_path = ROOT_DATASET / scenario / "data.yaml"

    if not best_weight.exists():
        print(f"PERIKSA: Bobot {best_weight} belum ada. Jalankan training terlebih dahulu.")
        continue

    print(f"\n{'='*30} EVALUASI TEST SET: {scenario} {'='*30}")
    model = YOLO(str(best_weight))

    # Evaluasi strictly pada split='test' dengan plots=True eksplisit
    test_metrics = model.val(
        data=str(yaml_path),
        split="test",
        project=str(TEST_EVALS_DIR),
        name=scenario,
        exist_ok=True,
        imgsz=GLOBAL_CONFIG["imgsz"],
        batch=GLOBAL_CONFIG["batch"],
        device=GLOBAL_CONFIG["device"],
        plots=True
    )

    # Metrik agregat keseluruhan
    mp = float(test_metrics.box.mp)
    mr = float(test_metrics.box.mr)
    map50 = float(test_metrics.box.map50)
    map50_95 = float(test_metrics.box.map)

    eval_records.append({
        "Skenario": scenario,
        "Keterangan": SCENARIO_AUG_CONFIGS[scenario]["keterangan"],
        "Precision": round(mp, 4),
        "Recall": round(mr, 4),
        "mAP50": round(map50, 4),
        "mAP50_95": round(map50_95, 4),
        "Path_Best_Weight": str(best_weight)
    })

    # Metrik per kelas (0: helmet, 1: no-helmet) menggunakan API resmi class_result()
    class_names = ["helmet", "no-helmet"]
    for idx_cls, cls_name in enumerate(class_names):
        try:
            p_cls, r_cls, map50_cls, map50_95_cls = test_metrics.box.class_result(idx_cls)
            per_class_records.append({
                "Skenario": scenario,
                "Kelas": cls_name,
                "Precision": round(float(p_cls), 4),
                "Recall": round(float(r_cls), 4),
                "mAP50": round(float(map50_cls), 4),
                "mAP50_95": round(float(map50_95_cls), 4)
            })
        except Exception as e:
            print(f"Peringatan ekstraksi kelas {cls_name}: {e}")

df_eval = pd.DataFrame(eval_records)
csv_save_path = EXPERIMENT_OUTPUT / "rekap_metrik_test_M0_M4.csv"
df_eval.to_csv(csv_save_path, index=False)

df_per_class = pd.DataFrame(per_class_records)
csv_per_class_path = EXPERIMENT_OUTPUT / "rekap_metrik_per_kelas_test.csv"
df_per_class.to_csv(csv_per_class_path, index=False)

print("\n" + "="*50)
print("REKAPITULASI METRIK EVALUASI TEST SET (M0 - M4)")
print("="*50)
print(df_eval.to_string(index=False))
print(f"\nFile rekapitulasi tersimpan di: {csv_save_path}")
print(f"File rekapitulasi per kelas tersimpan di: {csv_per_class_path}")



============================== EVALUASI TEST SET: M0 ==============================
Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,950 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2857.0±883.5 MB/s, size: 281.0 KB)
val: Scanning /content/eksperimen_final_M0_M4/M0/test/labels... 135 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 135/135 1.9Kit/s 0.1s
val: New cache created: /content/eksperimen_final_M0_M4/M0/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.8it/s 4.9s
                   all        135        479       0.89      0.716      0.811      0.404
                helmet         93        291      0.855      0.784      0.829      0.394
             no-helmet        113        188      0.926      0.649      0.793      0.415
Speed: 5.1ms preprocess, 6.4ms inference, 0.0ms loss, 3

## Bagian 16: Rekapitulasi Tabel dan Visualisasi Komparasi Metrik Aktual

> [!NOTE]
> **STATUS: MENUNGGU HASIL EKSPERIMEN.**
> Sel ini membaca berkas CSV riil yang dihasilkan dari evaluasi Bagian 15, memvisualisasikan grafik perbandingan performa dalam bentuk diagram batang, dan menampilkan kelima matriks konfusi secara berdampingan.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

csv_path = EXPERIMENT_OUTPUT / "rekap_metrik_test_M0_M4.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"File hasil evaluasi belum ada: {csv_path}. Jalankan Bagian 15 terlebih dahulu setelah training selesai.")

df = pd.read_csv(csv_path)

# 1. Tampilkan tabel komparasi
print("="*60)
print("TABEL PERBANDINGAN PERFORMA TEST SET (M0 s.d. M4)")
print("="*60)
display_cols = ["Skenario", "Precision", "Recall", "mAP50", "mAP50_95", "Keterangan"]
try:
    print(df[display_cols].to_markdown(index=False))
except Exception:
    print(df[display_cols].to_string(index=False))

# 2. Visualisasi Grafik Komparasi Bar Chart
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

metrics_to_plot = ["Precision", "Recall", "mAP50", "mAP50_95"]
df_melted = df.melt(id_vars=["Skenario"], value_vars=metrics_to_plot, var_name="Metrik", value_name="Nilai")

ax = sns.barplot(data=df_melted, x="Skenario", y="Nilai", hue="Metrik", palette="viridis")
plt.title("Perbandingan Performa Model YOLOv8n pada Test Set (M0 s.d. M4)", fontsize=14, pad=15)
plt.xlabel("Skenario Augmentasi", fontsize=12)
plt.ylabel("Skor Metrik", fontsize=12)
plt.ylim(0, 1.05)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Metrik")

# Beri label nilai di atas setiap batang grafik
for p in ax.patches:
    val = p.get_height()
    if val > 0:
        ax.annotate(f"{val:.3f}", (p.get_x() + p.get_width() / 2., val),
                    ha="center", va="bottom", fontsize=8, rotation=45, xytext=(0, 3),
                    textcoords="offset points")

plt.tight_layout()
plot_path = EXPERIMENT_OUTPUT / "grafik_perbandingan_metrik.png"
plt.savefig(plot_path, dpi=300)
plt.show()

print(f"\nGrafik perbandingan berhasil disimpan ke: {plot_path}")

# 3. Menampilkan Confusion Matrix dari setiap skenario berdampingan
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
scenarios = ["M0", "M1", "M2", "M3", "M4"]

for idx, scn in enumerate(scenarios):
    cm_path = TEST_EVALS_DIR / scn / "confusion_matrix.png"
    if cm_path.exists():
        img = plt.imread(str(cm_path))
        axes[idx].imshow(img)
        axes[idx].set_title(f"Confusion Matrix {scn}", fontsize=12)
        axes[idx].axis("off")
    else:
        axes[idx].text(0.5, 0.5, f"CM {scn}\nBelum Ada", ha="center", va="center")
        axes[idx].axis("off")

plt.tight_layout()
cm_combined_path = EXPERIMENT_OUTPUT / "confusion_matrix_all_scenarios.png"
plt.savefig(cm_combined_path, dpi=300)
plt.show()

print(f"Visualisasi gabungan Confusion Matrix tersimpan di: {cm_combined_path}")


# **M4 GRADIO**

In [ ]:
# ==============================================================================
# SCRIPT UTUH: WEB GRADIO + PENGUKUR LATENSI INFERENSI (YOLOv8 M4)
# ==============================================================================

# 1. Pastikan modul yang dibutuhkan terpasang
!pip install ultralytics gradio -q

import os
import time
import pandas as pd
from PIL import Image
import gradio as gr
from ultralytics import YOLO
from google.colab import drive

# 2. Hubungkan Google Drive jika belum
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# 3. Path Model M4 yang Tetap Dipertahankan
MODEL_PATH = '/content/drive/MyDrive/Penelitian_YOLOv8/hasil_eksperimen_final_M0_M4/training_runs/M4/weights/best.pt'

print(f">>> Memuat model M4 dari {MODEL_PATH} ...")
model = YOLO(MODEL_PATH)
print(f">>> Model berhasil dimuat! Kelas: {model.names}")


# 4. Fungsi Inferensi dengan Pengukur Latensi
def detect_helmet(input_image, conf_threshold, iou_threshold, use_agnostic_nms):
    if input_image is None:
        return None, "⚠️ Silakan unggah gambar terlebih dahulu.", pd.DataFrame()

    # Catat waktu awal (start)
    start_time = time.time()

    # Jalankan prediksi dengan resolusi standar imgsz=640
    results = model.predict(
        source=input_image,
        imgsz=640,
        conf=conf_threshold,
        iou=iou_threshold,
        agnostic_nms=use_agnostic_nms,
        verbose=False
    )
    result = results[0]

    # Catat waktu selesai (end)
    end_time = time.time()
    total_time_ms = (end_time - start_time) * 1000
    fps = 1.0 / (end_time - start_time) if (end_time - start_time) > 0 else 0

    # Ambil metrik latensi bawaan YOLOv8
    speed = result.speed
    preprocess_ms = speed.get('preprocess', 0.0)
    inference_ms = speed.get('inference', 0.0)
    postprocess_ms = speed.get('postprocess', 0.0)

    # Visualisasi bounding box rapi (line_width=2)
    annotated_bgr = result.plot(line_width=2)
    annotated_rgb = annotated_bgr[:, :, ::-1]

    detected_list = []
    count_helmet = 0
    count_no_helmet = 0

    if result.boxes is not None and len(result.boxes) > 0:
        for idx, box in enumerate(result.boxes, start=1):
            cls_id = int(box.cls[0].item())
            cls_name = model.names.get(cls_id, f"ID {cls_id}")
            conf_val = float(box.conf[0].item())

            if cls_name.lower() == 'helmet':
                count_helmet += 1
            elif cls_name.lower() == 'no-helmet':
                count_no_helmet += 1

            detected_list.append({
                "No": idx,
                "Label Kelas": cls_name,
                "Confidence Score": round(conf_val, 4),
                "Persentase": f"{conf_val * 100:.1f}%"
            })

    if detected_list:
        detected_list = sorted(detected_list, key=lambda x: x["Confidence Score"], reverse=True)
        for i, item in enumerate(detected_list, 1):
            item["No"] = i

    total_detected = len(detected_list)

    # Ringkasan hasil deteksi + Data Latensi untuk Laporan Penelitian
    ringkasan_md = f"""
### 📊 Ringkasan Deteksi
- **Total Objek Valid:** {total_detected}
  - 🟢 **Helmet (Menggunakan Helm):** {count_helmet}
  - 🔴 **No-Helmet (Tidak Menggunakan Helm):** {count_no_helmet}

---
### ⚡ Metrik Latensi Pemrosesan:
- **Total Waktu Proses:** `{total_time_ms:.1f} ms` (~`{fps:.1f} FPS`)
  - *Pre-processing:* `{preprocess_ms:.1f} ms`
  - *Inference YOLOv8 M4:* `{inference_ms:.1f} ms`
  - *Post-processing (NMS):* `{postprocess_ms:.1f} ms`

> *Catatan: Waktu inferensi murni GPU/CPU diukur dari pemanggilan model, sangat berguna untuk analisis performa komputasi pada skripsi.*
    """

    df_details = pd.DataFrame(detected_list) if detected_list else pd.DataFrame({"Informasi": ["Tidak ada objek yang terdeteksi."]})
    return annotated_rgb, ringkasan_md, df_details


# 5. Desain UI & Jalankan Server Gradio
gr.close_all()  # Tutup sesi Gradio lama

with gr.Blocks(title="Deteksi Helm Pengendara Sepeda Motor") as demo:
    gr.Markdown("# 🏍️ Deteksi Helm Pengendara Sepeda Motor (YOLOv8 M4)")

    with gr.Row():
        with gr.Column(scale=1):
            input_image_comp = gr.Image(label="Unggah Foto Pengendara Sepeda Motor", type="pil")
            conf_slider = gr.Slider(minimum=0.05, maximum=1.0, value=0.25, step=0.05, label="Confidence Threshold", info="Default: 0.25")

            with gr.Accordion("⚙️ Pengaturan NMS", open=True):
                iou_slider = gr.Slider(minimum=0.1, maximum=0.9, value=0.45, step=0.05, label="IoU Threshold", info="Default: 0.45")
                agnostic_cb = gr.Checkbox(value=True, label="Class-Agnostic NMS (Cegah tumpang tindih kelas)")

            btn_detect = gr.Button("🔍 Jalankan Deteksi", variant="primary", size="lg")

        with gr.Column(scale=1):
            output_image_comp = gr.Image(label="Hasil Deteksi", type="numpy")
            summary_output = gr.Markdown(value="*Unggah gambar lalu klik Jalankan Deteksi.*")
            table_output = gr.Dataframe(headers=["No", "Label Kelas", "Confidence Score", "Persentase"], label="Rincian Objek")

    btn_detect.click(
        fn=detect_helmet,
        inputs=[input_image_comp, conf_slider, iou_slider, agnostic_cb],
        outputs=[output_image_comp, summary_output, table_output]
    )

print("\n>>> Memulai server web Gradio...")
demo.launch(share=True, debug=True)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Mounted at /content/drive
>>> Memuat model M4 dari /content/drive/MyDrive/Penelitian_YOLOv8/hasil_eksperimen_final_M0_M4/training_runs/M4/weights/best.pt ...
>>> Model berhasil dimuat! Kelas: {0: 'helmet', 1: 'no-helmet'}

>>> Memulai server web Gradio...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b2519e3af40a0c536f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
